<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">业务应用程序</h2>
            <span style="color:#181;">对话助理是 Gen AI 非常常见的用例，前沿模型擅长细致入微的多轮对话。Gradio 让用户界面设计变得很容易。另一项关键技能是：用提示（prompt）提供上下文、信息与示例。
<br/><br/>
请思考如何把 AI 助手应用到你的业务中，并自己做一个原型：用系统提示（system prompt）给出业务背景，并为 LLM 定下语气与策略。</span>
        </td>
    </tr>
</table>


## 业务用例：电子商店助理

用本地 Ollama（`llama3.2:1b`）+ Gradio `ChatInterface` 做一家**电子产品店**的流式导购：系统提示里写清折扣与推荐策略；当用户消息含 `charger` 时，再动态追加一句促销提示（演示「运行时改 system」）。


In [ ]:
# ========== 电子商店导购：Ollama + Gradio ChatInterface（流式） ==========

# 导入 gradio：ChatInterface 一键生成多轮聊天 UI
import gradio as gr
# 从 openai 导入 OpenAI：走兼容协议调本地 Ollama
from openai import OpenAI

# 奥拉玛客户端：base_url 指向本机 /v1；api_key 占位即可
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# 选用的本地模型名（需事先 ollama pull）；字符串保持原样
MODEL = "llama3.2:1b"


# system_message：店铺导购人设 + 折扣规则 + 推荐策略（英文 prompt 必须原样保留）
system_message = """You are a helpful assistant in an electronics store.
You should gently guide customers toward products on sale.

Laptops are 20% off, smartphones are 15% off, and accessories are 40% off.

If a customer is unsure, recommend laptops as they provide the best value.

If the customer asks for gaming consoles:
- Inform them consoles are not on sale
- Suggest laptops or accessories instead

Encourage customers to explore discounted accessories.
"""

# chat：Gradio ChatInterface 回调；message 为当前用户输入，history 为历史 messages
def chat(message, history):
    # 每轮都以 system 开头；后面再拼历史与本轮 user
    messages = [{"role": "system", "content": system_message}]

    # 若用户提到 charger：运行时追加促销提示（演示动态改 system，不改写原 prompt 常量）
    if "charger" in message.lower():
        messages[0]["content"] += " Highlight that chargers and accessories are 40% off."

    # 把 Gradio 传来的历史对话接上（type="messages" 时为 role/content 字典列表）
    messages.extend(history)

    # 追加本轮用户消息
    messages.append({"role": "user", "content": message})

    # 流式 Chat Completions：边生成边 yield 累计文本
    stream = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    # response：累积助手回复；供界面打字机刷新
    response = ""
    for chunk in stream:
        # delta 可能为空，用 or "" 避免把 None 拼进字符串
        delta = chunk.choices[0].delta.content or ""
        response += delta
        yield response

# ChatInterface：type="messages" 表示 history 用 OpenAI 风格消息列表
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="🔌 AI Electronics Store Assistant",
    description="Ask about gadgets, deals, and recommendations!"
).launch()


## 这说明了什么

- **用提示工程（prompt engineering）承载业务规则**：折扣、缺货引导、默认推荐都写在 system 里
- **运行时动态追加促销**：检测到 `charger` 就改 system，类似电商里的情境营销
- **情境感知建议**：结合历史与当前问题，持续用同一套店铺策略做导购
